<div class = "alert alert-info"> <h2 align = "center"> <b> Cross-asset Macro Overlay </b> </h2> 
<h3 align = center> <font  color = "black"> News-to-portfolio attribution <font/> </h3>

</div>

### __Contexte__

Les portefeuilles multi-actifs sont traditionnellement construits sur des fondations quantitatives (corrélations, volatilités, betas). Cependant, une grande partie des mouvements de marché est macro-driven (inflation, taux, croissance, politique monétaire).

Deux sources d’information sont cruciales mais souvent exploitées séparément :

- Marché des options de taux → anticipe les mouvements futurs des banques centrales et l’incertitude (via vol implicite, skew, term structure).

- Flux de nouvelles macroéconomiques → messages textuels des banques centrales, données économiques, événements géopolitiques.

__Le projet vise à fusionner ces deux sources pour générer un overlay dynamique cross-asset, capable de repositionner le portefeuille face aux chocs macro.__



### __Objectifs principaux :__

- Construire un overlay dynamique cross-asset basé sur : Signaux textuels macro (news, communiqués, discours de banques centrales); Options rates (volatilité implicite, skew, convexité).

- Ajuster l’allocation en fonction de régimes de marché identifiés via Markov Switching / State-space.

- Quantifier l’impact des news sur la performance du portefeuille (news-to-portfolio attribution).



### __Attribution "News - to - Portfolio"__

Cette partie du projet, se concentre sur l'analyse de la performance du portefeuille (P&L pour Profit & Loss) en attribuant les contributions spécifiques aux facteurs, en particulier aux signaux issus des news macro (textuelles).

Cela permet une attribution granulaire, évitant de traiter le P&L comme une "boîte noire". En isolant l'impact des news, on quantifie comment les communiqués macro affectent directement la performance, aligné sur l'objectif principal du projet (quantifier l'impact des news sur le portefeuille : "News-to-portfolio attribution"). 


Dans un contexte financier, l'attribution est essentielle pour expliquer aux investisseurs pourquoi le P&L varie. 



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import statsmodels.api as sm

# Standard reproducibility seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)


# Load the model results from previous step (includes port_returns, weights, signals, regimes)
df = pd.read_parquet('./data/model_results.parquet')

# Assume key columns exist: 'optimized_weights' (list of weights), obs_vars (assets), 'port_returns' (daily portfolio returns)
# Signals: 'filtered_macro_signal' (news/macro), 'filtered_rates_signal' (rates/options)
# For attribution, focus on macro_signal as "news" proxy
obs_vars = ['^GSPC', '^STOXX50E', '^IRX', '^TNX', 'EURUSD=X', 'JPYUSD=X', 'CL=F', 'GC=F']

# Ensure port_returns is computed if not saved (from previous)
if 'port_returns' not in df.columns:
    df['port_returns'] = pd.Series(np.sum(df[obs_vars].values * np.array(df['optimized_weights'].tolist()), axis=1), index=df.index)


    

In [ ]:
# 5a) Décomposition P&L
# delta_portfolio = weights_t * returns_t → Already as port_returns (daily P&L proxy, assuming normalized)
# Contribution de chaque facteur: Use regression-based attribution
# Regress port_returns ~ macro_signal + rates_signal + controls (e.g., lag port)
# Then, contribution_t = beta_factor * factor_t (partial effect)

# Prepare data for regression
df_attrib = df[['port_returns', 'filtered_macro_signal', 'filtered_rates_signal']].dropna()
df_attrib['lag_port_returns'] = df_attrib['port_returns'].shift(1).fillna(0)  # Control for momentum

# Fit OLS model
X = sm.add_constant(df_attrib[['filtered_macro_signal', 'filtered_rates_signal', 'lag_port_returns']])
y = df_attrib['port_returns']
model_attrib = sm.OLS(y, X).fit()

# Print summary
print("P&L Attribution Regression Summary:")
print(model_attrib.summary())

# Compute contributions
betas = model_attrib.params
df_attrib['contrib_macro'] = betas['filtered_macro_signal'] * df_attrib['filtered_macro_signal']  # news_impact
df_attrib['contrib_rates'] = betas['filtered_rates_signal'] * df_attrib['filtered_rates_signal']
df_attrib['contrib_other'] = df_attrib['port_returns'] - (df_attrib['contrib_macro'] + df_attrib['contrib_rates'])  # Residual incl. lag/const

# Cumulative contributions
df_attrib['cum_contrib_macro'] = df_attrib['contrib_macro'].cumsum()
df_attrib['cum_contrib_rates'] = df_attrib['contrib_rates'].cumsum()
df_attrib['cum_contrib_other'] = df_attrib['contrib_other'].cumsum()

# news_impact specific: contribution(delta_portfolio, macro_signal) ≈ contrib_macro
print("\nSample News Impact (contrib_macro):")
print(df_attrib['contrib_macro'].head())



In [ ]:
# 5b) Visualisation
# Heatmap: score macro vs allocation (e.g., macro_score bins vs average weights per asset)
# Bin macro_signal into quantiles (e.g., 5 bins: very dovish to very hawkish)
df['macro_bin'] = pd.qcut(df['filtered_macro_signal'], q=5, labels=['Very Dovish', 'Dovish', 'Neutral', 'Hawkish', 'Very Hawkish'])

# Average weights per bin and asset
weights_df = pd.DataFrame(df['optimized_weights'].tolist(), index=df.index, columns=obs_vars)
weights_df['macro_bin'] = df['macro_bin']
avg_weights = weights_df.groupby('macro_bin')[obs_vars].mean()

# Heatmap with seaborn
plt.figure(figsize=(12, 6))
sns.heatmap(avg_weights, annot=True, cmap='YlGnBu', fmt='.2f')
plt.title('Heatmap: Macro Score (Bins) vs Average Allocation Weights')
plt.xlabel('Assets')
plt.ylabel('Macro Signal Bins')

plt.savefig("./output/macro_score_vs_alloc.png", dpi = 300, bbox_inches = 'tight')
plt.show()

# Interactive with plotly
fig = px.imshow(avg_weights.values, x=obs_vars, y=avg_weights.index,
                labels=dict(color='Weight'), title='Interactive Heatmap: Macro Score vs Allocation')
fig.show()



In [ ]:
# Bar Charts: Contribution des news au P&L
# Monthly aggregation for bars (cumulative or average)
df_monthly = df_attrib.resample('M').agg({
    'contrib_macro': 'sum',  # Cumulative monthly contribution
    'contrib_rates': 'sum',
    'contrib_other': 'sum'
})

# Stacked bar chart with matplotlib
df_monthly.plot(kind='bar', stacked=True, figsize=(14, 7))
plt.title('Bar Chart: Monthly P&L Contributions (News/Macro, Rates, Other)')
plt.xlabel('Month')
plt.ylabel('Contribution to P&L')
plt.legend(title='Factors')
plt.xticks(rotation=45)
plt.savefig("./output/monthly_pnl_contrib.png", dpi = 300, bbox_inches = 'tight')

plt.show()

# Interactive with plotly
fig_bar = px.bar(df_monthly.reset_index(), x='index', y=['contrib_macro', 'contrib_rates', 'contrib_other'],
                 title='Interactive Bar Chart: Contributions to P&L',
                 labels={'value': 'Contribution', 'index': 'Month', 'variable': 'Factor'})
fig_bar.update_layout(barmode='stack')
fig_bar.show()



In [ ]:
# Focus on news_impact: Separate bar for macro contrib
plt.figure(figsize=(12, 6))
df_monthly['contrib_macro'].plot(kind='bar', color='skyblue')
plt.title('Bar Chart: Contribution des News (Macro Signal) au P&L Mensuel')
plt.xlabel('Month')
plt.ylabel('News Impact on P&L')
plt.xticks(rotation=45)
plt.savefig("./output/macro_contrib.png", dpi = 300, bbox_inches = 'tight')

plt.show()



In [ ]:
# Save attribution results
df_attrib.to_parquet('./data/attribution_results.parquet')
print("Attribution “News-to-Portfolio” completed and saved.")